In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,make_scorer
from sklearn.model_selection import RandomizedSearchCV

In [5]:
df=pd.read_csv('data/pre_process_data.csv')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7021 entries, 0 to 7020
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7021 non-null   object 
 1   SeniorCitizen     7021 non-null   int64  
 2   Partner           7021 non-null   object 
 3   Dependents        7021 non-null   object 
 4   tenure            7021 non-null   int64  
 5   PhoneService      7021 non-null   object 
 6   MultipleLines     7021 non-null   object 
 7   InternetService   7021 non-null   object 
 8   OnlineSecurity    7021 non-null   object 
 9   OnlineBackup      7021 non-null   object 
 10  DeviceProtection  7021 non-null   object 
 11  TechSupport       7021 non-null   object 
 12  StreamingTV       7021 non-null   object 
 13  StreamingMovies   7021 non-null   object 
 14  Contract          7021 non-null   object 
 15  PaperlessBilling  7021 non-null   object 
 16  PaymentMethod     7021 non-null   object 


In [24]:
df['MonthlyCharges'].unique()

array([29.85, 56.95, 53.85, ..., 63.1 , 44.2 , 78.7 ])

In [103]:
X=df.drop(columns=['Churn'],axis=1)
y=df['Churn']

In [104]:
num_faeture=X.select_dtypes(exclude='object').columns
cat_feature=X.select_dtypes(include="object").columns

In [105]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

numerical_feature=StandardScaler()
categorical_feature=OneHotEncoder(drop='first')


preprocessor=ColumnTransformer(
    [
        ('onehotencoder',categorical_feature,cat_feature),
        ('StandardScaler',numerical_feature,num_faeture)
    ],remainder='passthrough'
)

In [106]:
X=preprocessor.fit_transform(X)

In [107]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,-0.440508,-1.282728,-1.164135,-0.997328
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,-0.440508,0.062387,-0.262811,-0.176347
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-0.440508,-1.241967,-0.365914,-0.962760
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,-0.440508,0.510759,-0.750058,-0.197869
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,-0.440508,-1.241967,0.194503,-0.943556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7016,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,-0.440508,-0.345224,0.663458,-0.131759
7017,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,-0.440508,1.611307,1.275428,2.239996
7018,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,-0.440508,-0.875118,-1.172450,-0.857558
7019,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,2.270104,-1.160445,0.317562,-0.875151


In [108]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

In [109]:
def evaluate(true,pred):
    AccuracyScore=accuracy_score(true,pred)
    f1Score=f1_score(true,pred,pos_label='Yes')
    precisionScore=precision_score(true,pred,pos_label='Yes')
    recallScore=recall_score(true,pred,pos_label='Yes')
    
    return AccuracyScore,f1Score,precisionScore,recallScore

In [110]:
models={
    "RandomForestClassifier":RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=42),
    "GradientBoostingClassifier":GradientBoostingClassifier(random_state=42),
    "DecisionTreeClassifier":DecisionTreeClassifier(class_weight='balanced',random_state=42),
    'LogisticRegression':LogisticRegression(class_weight='balanced',
                                            max_iter=1000,
                                            random_state=42),
    "KNeighborsClassifier":KNeighborsClassifier(),
    "SVC":SVC(class_weight='balanced',probability=True,random_state=42)
}

In [111]:
model_list=[]
ac_list=[]
train_ac_list=[]
train_f1_list = []
test_f1_list = []
train_recall=[]
test_recall=[]

In [112]:
for i in range(len(models)):
    model=list(models.values())[i]
    model.fit(X_train,y_train)
    
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)
    
    y_train_acc,y_train_f1,y_train_precision,y_train_recall=evaluate(y_train,y_train_pred)
    y_test_acc,y_test_f1,y_test_precision,y_test_recall=evaluate(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    ac_list.append(y_test_acc)
    train_ac_list.append(y_train_acc)
    train_f1_list.append(y_train_f1)
    test_f1_list.append(y_test_f1)
    train_recall.append(y_train_recall)
    test_recall.append(y_test_recall)
    
    print('-------------------------------')
    
    print("Train performance")
    print('accracy score:{:.4f}'.format(y_train_acc))
    print('f1 score:{:.4f}'.format(y_train_f1))
    print('precision score :{:.4f}'.format(y_train_precision))
    print('recall score :{:.4f}'.format(y_train_recall))
    
    print('\n')
    
    print("Test performance")
    print('accracy score:{:.4f}'.format(y_test_acc))
    print('f1 score:{:.4f}'.format(y_test_f1))
    print('precision score :{:.4f}'.format(y_test_precision))
    print('recall score :{:.4f}'.format(y_test_recall))

RandomForestClassifier
-------------------------------
Train performance
accracy score:0.9977
f1 score:0.9957
precision score :0.9915
recall score :1.0000


Test performance
accracy score:0.7899
f1 score:0.5370
precision score :0.6426
recall score :0.4612
GradientBoostingClassifier
-------------------------------
Train performance
accracy score:0.8249
f1 score:0.6312
precision score :0.7127
recall score :0.5664


Test performance
accracy score:0.8109
f1 score:0.5951
precision score :0.6854
recall score :0.5259
DecisionTreeClassifier
-------------------------------
Train performance
accracy score:0.9977
f1 score:0.9957
precision score :0.9915
recall score :1.0000


Test performance
accracy score:0.7329
f1 score:0.4771
precision score :0.4942
recall score :0.4612
LogisticRegression
-------------------------------
Train performance
accracy score:0.7525
f1 score:0.6361
precision score :0.5206
recall score :0.8177


Test performance
accracy score:0.7432
f1 score:0.6188
precision score :0.50

In [113]:
pd.DataFrame(
    list(zip(
        model_list,
        ac_list,
        train_ac_list,
        train_f1_list,
        test_f1_list,
        train_recall,
        test_recall
    )),
    columns=[
        'models',
        'testing_accuracy',
        'train_accuracy',
        'train_f1',
        'test_f1',
        "train_recall",
        "test_recall"
    ]
).sort_values(
    by=["test_recall","test_f1",'testing_accuracy'],
    ascending=False
)

,models,testing_accuracy,train_accuracy,train_f1,test_f1,train_recall,test_recall
3,LogisticRegression,0.743166,0.752517,0.636135,0.618766,0.817660,0.788793
5,SVC,0.748861,0.761633,0.644979,0.622108,0.818378,0.782328
1,GradientBoostingClassifier,0.810934,0.824881,0.631200,0.595122,0.566403,0.525862
4,KNeighborsClassifier,0.756834,0.837797,0.682999,0.520763,0.660445,0.500000
0,RandomForestClassifier,0.789863,0.997721,0.995711,0.537014,1.000000,0.461207
2,DecisionTreeClassifier,0.732916,0.997721,0.995711,0.477146,1.000000,0.461207


In [114]:
lr_params = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "max_iter": [200, 500, 1000],
    "class_weight": [None, "balanced"]
}

In [115]:
randomCV_model=[
    ('LG',LogisticRegression(),lr_params)
]

In [116]:
model_params={}

In [117]:
f1_scorer = make_scorer(f1_score, pos_label='Yes')

for name, model, params in randomCV_model:

    Rc = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=50,
        scoring=f1_scorer,
        cv=3,
        verbose=2,
        random_state=10,
        n_jobs=-1
    )

    Rc.fit(X_train, y_train)

    model_params[name] = Rc.best_params_

for model_name in model_params:
    print(f"___best param for {model_name}__")
    print(model_params[model_name])

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END C=0.1, class_weight=balanced, max_iter=200, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END C=0.1, class_weight=balanced, max_iter=200, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.1, class_weight=balanced, max_iter=200, penalty=l1, solver=liblinear; total time=   0.1s


/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END C=10, class_weight=None, max_iter=200, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=10, class_weight=None, max_iter=200, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=10, class_weight=None, max_iter=200, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=1, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   0.3s
[CV] END C=1, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   0.3s
[CV] END C=10, class_weight=balanced, max_iter=1000, penalty=l2, solver=saga; total time=   1.0s


/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END C=10, class_weight=None, max_iter=500, penalty=l2, solver=saga; total time=   1.2s[CV] END C=10, class_weight=None, max_iter=500, penalty=l2, solver=saga; total time=   1.2s

[CV] END C=0.001, class_weight=None, max_iter=1000, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=0.001, class_weight=None, max_iter=1000, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=0.001, class_weight=None, max_iter=1000, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=200, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END C=1, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=0.01, class_weight=None, max_iter=200, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END C=100, class_weight=None, max_iter=200, penalty=l1, solver=liblinear; total time=   0.4s
[CV] END C=100, class_weight=None, max_iter=200, penalty=l1, solver=liblinear; total time=   0.5s
[CV] END C=0.01, class_weig

...

[CV] END C=100, class_weight=None, max_iter=200, penalty=l1, solver=saga; total time=   0.6s
[CV] END C=100, class_weight=balanced, max_iter=500, penalty=l1, solver=saga; total time=   1.3s
[CV] END C=10, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   0.2s
[CV] END C=100, class_weight=balanced, max_iter=200, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=100, class_weight=balanced, max_iter=200, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=0.1, class_weight=balanced, max_iter=500, penalty=l2, solver=saga; total time=   0.1s
[CV] END C=0.1, class_weight=balanced, max_iter=500, penalty=l2, solver=saga; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=200, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=200, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=200, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=0.1, class_weight=

/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END C=100, class_weight=balanced, max_iter=200, penalty=l2, solver=saga; total time=   0.5s
[CV] END C=10, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   0.6s
[CV] END C=0.001, class_weight=balanced, max_iter=1000, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END C=0.1, class_weight=None, max_iter=500, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.1, class_weight=None, max_iter=500, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END C=1, class_weight=None, max_iter=500, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.001, class_weight=balanced, max_iter=1000, penalty=l1, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=500, penalty=l1, solver=saga; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=500, penalty=l1, solver=saga; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=500, penalty=l1, solver=saga; total time=   0.1s
[CV] END C=0.1, class_w

/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END C=100, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   1.5s
[CV] END C=100, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   1.5s
[CV] END C=100, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   1.5s
___best param for LG__
{'solver': 'saga', 'penalty': 'l1', 'max_iter': 200, 'class_weight': 'balanced', 'C': 0.1}


In [118]:
models = {
    "Logistic Regression": LogisticRegression(
        solver='saga',
        penalty='l1',
        max_iter=200,
        class_weight='balanced',
        C=10
    )
}

In [119]:
for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_acc, y_train_f1, y_train_precision, y_train_recall = evaluate(
        y_train, y_train_pred
    )

    y_test_acc, y_test_f1, y_test_precision, y_test_recall = evaluate(
        y_test, y_test_pred
    )

    print('-------------------------------')

    print("Train performance")
    print('accuracy score:{:.8f}'.format(y_train_acc))
    print('f1 score:{:.8f}'.format(y_train_f1))
    print('precision score:{:.8f}'.format(y_train_precision))
    print('recall score:{:.8f}'.format(y_train_recall))

    print('\n')

    print("Test performance")
    print('accuracy score:{:.8f}'.format(y_test_acc))
    print('f1 score:{:.8f}'.format(y_test_f1))
    print('precision score:{:.8f}'.format(y_test_precision))
    print('recall score:{:.8f}'.format(y_test_recall))

-------------------------------
Train performance
accuracy score:0.75118708
f1 score:0.63469046
precision score:0.51892385
recall score:0.81694185


Test performance
accuracy score:0.74202733
f1 score:0.61772152
precision score:0.50762829
recall score:0.78879310
